# TweetAge 2026 PUJ · Grupo 4 · Modelo final `greedy_kn`

**Tarea.** Predecir el rango de edad (13-17, 18-24, 25-34, 35-49, 50-64, 65-xx) del autor de **un solo tuit** en español. Métrica: **F1 macro**.

**Reglas que cumple este notebook.** Solo se usan **modelos de lenguaje propios** (n-gramas de caracteres y de palabras con conteos por clase), entrenados únicamente con `age_train.csv`. No hay transformers, embeddings, Word2Vec, GloVe, TF-IDF, vectorización de documentos ni modelos preentrenados. Los arreglos de numpy solo guardan **conteos por clase** y suman log-probabilidades.

**Resultado.** F1 macro 0.5992 en validación local y **0.61369 en el leaderboard público** (la mezcla inicial del equipo daba 0.60021).

**Qué hace el notebook, de principio a fin y sin archivos externos:**

1. Carga los datos y resume lo más importante del análisis exploratorio.
2. Define el preprocesamiento y los modelos de lenguaje.
3. **Valida** tres modelos en una partición sin fuga: la mezcla inicial, `greedy_ent` y `greedy_kn`. Muestra el F1 por clase, la matriz de confusión y ejemplos de errores.
4. Entrena las dos submissions finales (`greedy_kn` y `greedy_ent`) con **todo** el train y escribe un CSV por modelo.

**Tiempo aproximado** (4 CPU, 16–30 GB de RAM): validación 20–30 min y entrenamiento final 20–30 min. Lo pesado son los LMs Kneser-Ney de caracteres (~2.6 GB cada uno). Las probabilidades de cada modelo se guardan en `cache/` dentro de la carpeta de trabajo: si se vuelve a ejecutar en la misma sesión, se cargan en segundos.

El detalle de la exploración, los experimentos descartados y el análisis de errores completo están en el informe del grupo.

## 0. Configuración

- `VALIDAR = False` salta la sección 3 y solo entrena los modelos finales y genera los CSV.
- `N_JOBS` = cuántos modelos se entrenan en paralelo. Cada proceso entrena un modelo completo, así que la memoria crece con `N_JOBS`. Con 2 cabe en Kaggle (30 GB); con 16 GB de RAM usar 1.

In [ ]:
import gc
import hashlib
import html
import math
import os
import re
import sys
import time
from array import array
from functools import lru_cache
from pathlib import Path

import matplotlib.pyplot as plt
import nltk
import numpy as np
import pandas as pd
import scipy
import sklearn
from nltk.tokenize import TweetTokenizer
from sklearn.model_selection import StratifiedGroupKFold

VALIDAR = True  # False: solo entrenamiento final + submission
N_JOBS = 8  # modelos entrenados en paralelo (memoria ≈ 3 GB por proceso en los KN)
SEMILLA_VALIDACION = 20261109  # partición de "confirmación" usada en el informe
SEMILLA_EJEMPLOS = 7  # sorteo de ejemplos de error
N_BOOT = 1000  # remuestreos del intervalo bootstrap
MUESTRA = int(os.environ.get("TWEETAGE_MUESTRA", "0"))  # solo para pruebas rápidas: 0 = todos los datos

CLASES = ["13-17", "18-24", "25-34", "35-49", "50-64", "65-xx"]
pd.set_option("display.max_colwidth", 110)

print("Python", sys.version.split()[0])
for lib in (np, pd, sklearn, scipy, nltk):
    print(f"{lib.__name__:<8} {lib.__version__}")

In [ ]:
def entorno():
    if Path("/kaggle/input").exists():
        return "kaggle"
    if "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ:
        return "colab"
    return "local"


ENTORNO = entorno()
WORK = {"kaggle": Path("/kaggle/working"), "colab": Path("/content")}.get(ENTORNO, Path.cwd())
CACHE = WORK / "cache"
CACHE.mkdir(parents=True, exist_ok=True)


def buscar_csv(nombre):
    # Kaggle: datos de la competencia en /kaggle/input. Colab: /content o /content/data. Local: carpeta actual o data/.
    raices = {"kaggle": [Path("/kaggle/input")], "colab": [Path("/content")]}.get(
        ENTORNO, [Path.cwd(), Path.cwd().parent]
    )
    if os.environ.get("TWEETAGE_DATA"):
        raices = [Path(os.environ["TWEETAGE_DATA"])]
    for raiz in raices:
        hits = sorted(p for p in raiz.rglob(nombre) if "cache" not in p.parts)
        if hits:
            return hits[0]
    raise FileNotFoundError(
        f"No se encontró {nombre}. Agrega los datos de la competencia (o define TWEETAGE_DATA)."
    )


print("Entorno:", ENTORNO, "| carpeta de trabajo:", WORK)

## 1. Datos

Los CSV usan `;` como separador. El texto se limpia lo mínimo: los nulos pasan a `""` y se quitan los espacios de los extremos (varios tuits traen relleno de espacios).

In [ ]:
train = pd.read_csv(buscar_csv("age_train.csv"), sep=";", encoding="utf-8-sig")
test = pd.read_csv(buscar_csv("age_test.csv"), sep=";", encoding="utf-8-sig")
assert {"id", "text", "age_range"} <= set(train.columns) and {"id", "text"} <= set(test.columns)
if MUESTRA:  # solo para probar que el notebook corre de punta a punta
    train = train.sample(MUESTRA, random_state=0).reset_index(drop=True)
    test = test.head(MUESTRA // 4).reset_index(drop=True)


def textos(df):
    return df["text"].fillna("").astype(str).str.strip().tolist()


X, X_test = textos(train), textos(test)
y = train["age_range"].to_numpy()
print(f"train: {len(X):,} tuits ({len(set(X)):,} textos distintos) | test: {len(X_test):,}")

### 1.1 Lo más importante del análisis exploratorio

Se calculan cuatro rasgos por clase. Explican varias decisiones del modelo:

- **Balance.** Las clases están casi balanceadas, así que el F1 macro no castiga una clase pequeña.
- **Longitud.** Crece con la edad. Los LMs secuenciales normalizan por longitud para que no domine.
- **% de letras en mayúscula.** Tiene forma de U: los jóvenes y los mayores usan más mayúsculas. Los modelos iniciales pasaban todo a minúsculas y perdían esta señal; por eso el modelo final usa n-gramas de caracteres **con** mayúsculas.
- **% de RT y de tuits con mención.** Los mayores retuitean más. Las cuentas mencionadas son una señal social fuerte: el 76 % de las menciones de test ya aparece en train.

In [ ]:
s = pd.Series(X)
letras = s.str.count(r"[A-Za-z]").clip(lower=1)
eda = pd.DataFrame(
    {
        "tuits": pd.Series(y).value_counts(),
        "% del train": (100 * pd.Series(y).value_counts(normalize=True)).round(1),
        "mediana de caracteres": s.str.len().groupby(y).median(),
        "% letras en mayúscula": (
            100 * s.str.count(r"[A-Z]").groupby(y).sum() / letras.groupby(y).sum()
        ).round(1),
        "% RT": (100 * s.str.match(r"(?i)^rt @").groupby(y).mean()).round(1),
        "% con mención": (100 * s.str.contains(r"@\w+").groupby(y).mean()).round(1),
    }
).reindex(CLASES)
eda

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
eda["mediana de caracteres"].plot.bar(
    ax=axes[0], color="#4C72B0", rot=0, title="Mediana de caracteres por clase"
)
eda["% letras en mayúscula"].plot.bar(
    ax=axes[1], color="#DD8452", rot=0, title="% de letras en mayúscula (forma de U)"
)
axes[1].set_ylim(eda["% letras en mayúscula"].min() - 2, eda["% letras en mayúscula"].max() + 1)
for ax in axes:
    ax.set_xlabel("")
plt.tight_layout()
plt.show()

## 2. Preprocesamiento y modelos de lenguaje

### 2.1 Preprocesamiento

- **Canales de caracteres.** Trabajan sobre el texto tal cual. Las variantes `_cs` conservan mayúsculas; las demás pasan a minúsculas.
- **Canales de palabras.** Usan `TweetTokenizer` de NLTK (licencia Apache 2.0). Funciona solo con expresiones regulares (no tiene nada entrenado), mantiene emoticonos como `:)` y `<3`, decodifica entidades HTML (`&lt;3`) y recorta alargamientos (`siiiii` → `siii`). No se usa `word_tokenize`, que depende de Punkt (preentrenado).
- **Compuerta de menciones.** El canal de menciones con posición solo vota en los tuits que tienen alguna `@`.

In [ ]:
_TT = TweetTokenizer(preserve_case=False, reduce_len=True, strip_handles=False)


@lru_cache(maxsize=600_000)
def tweet_tokens(text):
    # Tokens de TweetTokenizer en minúsculas (cacheados: varios canales tokenizan los mismos tuits).
    return tuple(_TT.tokenize("" if text is None else str(text)))


def is_url(tok):
    t = tok.casefold()
    return t.startswith("http") or t.startswith("www.")


TOKEN_RE = re.compile(r"https?://\S+|www\.\S+|@\w+|#\w+|\w+|[^\w\s]", flags=re.UNICODE | re.IGNORECASE)


def _norm(text):
    return "" if text is None else str(text).casefold()


def mention_mask(textos):
    return np.array([bool(re.search(r"@\w+", t)) for t in textos], dtype=bool)


print(tweet_tokens("RT @Ana: jajajajaja te amooo &lt;3 :) http://t.co/x #Viernes"))

### 2.2 Modelos de lenguaje condicionales a la clase

Todos los modelos siguen la misma idea. Para cada clase $c$ se entrena un LM solo con los tuits de esa clase, y un tuit $x$ recibe el puntaje $\log P(x \mid c) + \log P(c)$. La clase con mayor puntaje gana. Los modelos solo se diferencian en **qué eventos** cuentan y **cómo suavizan** los eventos no vistos.

**Implementación.** En `fit`, cada evento (n-grama, mención, transición) recibe un id entero y sus conteos por clase se guardan en una tabla `(eventos × 6)` con los logaritmos precalculados. Al predecir, se busca el id de cada evento y se suman sus filas. Es la misma matemática de un Naive Bayes o un Markov escrito con diccionarios, solo que más rápida. No se construye ninguna representación vectorial del documento.

**a) Naive Bayes multinomial y modelos categóricos** (suavizado aditivo α):

- `NgramNB`: bolsa de n-gramas de caracteres o de palabras.
- `IdentityLM`: conjunto de menciones o hashtags del tuit.
- `PositionalWordLM`: palabras del inicio y del final del tuit.
- `PositionMentionLM`: menciones al inicio (respuesta) frente a menciones dentro del texto.
- `ShapeNB`: n-gramas de la "forma" del texto (`A` mayúscula, `a` minúscula, `9` dígito).

In [ ]:
CHUNK = 20_000  # textos por bloque al predecir (limita la memoria)


def _segment_sums(table, ids, lengths):
    # Suma, por texto, las filas table[ids] de cada segmento (textos vacíos -> 0).
    out = np.zeros((len(lengths), table.shape[1]), dtype=np.float64)
    nonempty = lengths > 0
    if ids.size:
        starts = np.concatenate(([0], np.cumsum(lengths)[:-1]))
        out[nonempty] = np.add.reduceat(table[ids], starts[nonempty], axis=0)
    return out


class _EventLM:
    # LM multinomial/categórico sobre eventos discretos con suavizado aditivo:
    # score_c = log P(c) - n·log(N_c + α·V) + Σ_e log(conteo_c(e) + α)

    def __init__(self, alpha):
        self.alpha = float(alpha)

    def _events(self, text):
        raise NotImplementedError

    def _extra_vocab(self):
        return 0

    def fit(self, X, y):
        y = np.asarray(y)
        self.classes_, sizes = np.unique(y, return_counts=True)
        class_idx = {c: i for i, c in enumerate(self.classes_)}
        k = len(self.classes_)
        self.index_ = {}
        index, flat = self.index_, array("q")
        for text, label in zip(X, y, strict=True):
            c = class_idx[label]
            setdef = index.setdefault
            flat.extend(setdef(e, len(index)) * k + c for e in self._events(text))
        n_events = len(index)
        counts = np.bincount(np.frombuffer(flat, dtype=np.int64), minlength=n_events * k).reshape(n_events, k)
        del flat
        self.totals_ = counts.sum(axis=0)
        self.vocab_size_ = n_events + self._extra_vocab()
        self.log_priors_ = np.log(sizes / len(y))
        # fila extra al final = evento no visto
        self.log_table_ = np.vstack([np.log(counts + self.alpha), np.full((1, k), math.log(self.alpha))])
        self.log_denom_ = np.log(self.totals_ + self.alpha * self.vocab_size_)
        return self

    def decision_function(self, X):
        X = list(X)
        unk, get = len(self.index_), self.index_.get
        out = np.empty((len(X), len(self.classes_)), dtype=np.float64)
        for s in range(0, len(X), CHUNK):
            block = X[s : s + CHUNK]
            ids, lengths = array("q"), np.empty(len(block), dtype=np.int64)
            for i, text in enumerate(block):
                ev = [get(e, unk) for e in self._events(text)]
                lengths[i] = len(ev)
                ids.extend(ev)
            ids = np.frombuffer(ids, dtype=np.int64)
            out[s : s + len(block)] = (
                self.log_priors_
                - lengths[:, None] * self.log_denom_
                + _segment_sums(self.log_table_, ids, lengths)
            )
        return out


class NgramNB(_EventLM):
    # Naive Bayes multinomial sobre n-gramas de caracteres ("char", con o sin mayúsculas) o de palabras ("word").

    def __init__(self, unit="char", orders=(5,), alpha=0.1, case_sensitive=False, tokenizer="regex"):
        super().__init__(alpha)
        self.unit, self.orders, self.case_sensitive, self.tokenizer = (
            unit,
            tuple(orders),
            case_sensitive,
            tokenizer,
        )

    def _events(self, text):
        if self.unit == "char":
            t = ("" if text is None else str(text)) if self.case_sensitive else _norm(text)
            return [t[i : i + n] for n in self.orders for i in range(len(t) - n + 1)]
        toks = tweet_tokens(text) if self.tokenizer == "tweet" else TOKEN_RE.findall(_norm(text))
        return [tuple(toks[i : i + n]) for n in self.orders for i in range(len(toks) - n + 1)]

    def _extra_vocab(self):
        return len(self.orders)


class IdentityLM(_EventLM):
    # Modelo categórico sobre el conjunto de ítems únicos del tuit (menciones o hashtags).

    def __init__(self, pattern, alpha=0.1):
        super().__init__(alpha)
        self.pattern = re.compile(pattern, flags=re.UNICODE)

    def _events(self, text):
        return list(set(self.pattern.findall(_norm(text))))

    def _extra_vocab(self):
        return 1


class PositionalWordLM(_EventLM):
    # Eventos (zona, token): el primer y el último cuarto de los tokens del tuit.

    def __init__(self, alpha=0.03, tokenizer="regex"):
        super().__init__(alpha)
        self.tokenizer = tokenizer

    def _events(self, text):
        tokens = tweet_tokens(text) if self.tokenizer == "tweet" else TOKEN_RE.findall(_norm(text))
        n = len(tokens)
        if not n:
            return []
        k = max(1, n // 4)
        k = min(k, n // 2) if n > 1 else 1
        return [("start", t) for t in tokens[:k]] + [("end", t) for t in tokens[n - k :]]

    def _extra_vocab(self):
        return len({e[0] for e in self.index_})


class PositionMentionLM(_EventLM):
    # Menciones distinguiendo la inicial (respuesta) de las que van dentro del texto.

    def __init__(self, alpha=0.01, tokenizer="regex"):
        super().__init__(alpha)
        self.tokenizer = tokenizer

    def _events(self, text):
        toks = tweet_tokens(text) if self.tokenizer == "tweet" else TOKEN_RE.findall(_norm(text))
        events = set()
        for i, tok in enumerate(toks):
            if tok.startswith("@") and (self.tokenizer == "regex" or len(tok) > 1):
                events.add(("leading" if i == 0 else "in_text", tok))
        return list(events)

    def _extra_vocab(self):
        return 2


_URL_RE, _MENTION_RE, _HASHTAG_RE = (
    re.compile(r"https?://\S+|www\.\S+"),
    re.compile(r"@\w+"),
    re.compile(r"#\w+"),
)


def shape(text, max_run=6):
    # Forma del texto: mayúscula->A, minúscula->a, dígito->9, espacio->' ', puntuación igual.
    # URLs, menciones y hashtags se reducen a un símbolo; las rachas se cortan en max_run.
    t = "" if text is None else str(text)
    t = _HASHTAG_RE.sub("#", _MENTION_RE.sub("@", _URL_RE.sub("\x01", t)))
    out, prev, run = [], None, 0
    for ch in t:
        s = (
            "A"
            if ch.isupper()
            else "a"
            if ch.islower()
            else "9"
            if ch.isdigit()
            else " "
            if ch.isspace()
            else ch
        )
        run = run + 1 if s == prev else 1
        prev = s
        if run <= max_run:
            out.append(s)
    return "".join(out)


class ShapeNB(NgramNB):
    def __init__(self, orders=(7,), alpha=0.1):
        super().__init__(unit="char", orders=orders, alpha=alpha)

    def _events(self, text):
        s = shape(text)
        return [s[i : i + n] for n in self.orders for i in range(len(s) - n + 1)]


print(shape("RT @Ana: JAJAJA 2015!! http://t.co/x"))

**b) Cadenas de Markov con suavizado aditivo.** Modelan el orden: $P(x_t \mid \text{contexto})$. El puntaje se divide por la longitud del tuit.

- `CharMarkov`: caracteres con un contexto de 2 o 4.
- `WordMarkov`: palabras de orden 3. Las URLs pasan a `URL` y las palabras raras a `<unk>`. Con `entity_min_count=5`, las menciones y hashtags que aparecen al menos 5 veces en train se conservan como palabras (son señal social) y el resto pasa a `@USER` / `#TAG`.

In [ ]:
class _SequenceLM:
    # LM de Markov condicional a la clase, normalizado por longitud:
    # score_c = (Σ_t log[(trans_c(ctx,x)+α) / (ctx_c(ctx)+α·V)] + log P(c)) / n

    def __init__(self, alpha):
        self.alpha = float(alpha)

    def fit(self, texts, y):
        texts, y = list(texts), np.asarray(y)
        self.classes_, sizes = np.unique(y, return_counts=True)
        self.log_priors_ = np.log(sizes / sizes.sum())
        class_idx = {c: i for i, c in enumerate(self.classes_)}
        k = len(self.classes_)
        self._prepare_vocab(texts)
        self.pair_index_, self.ctx_index_ = {}, {}
        p_idx, c_idx = self.pair_index_, self.ctx_index_
        p_flat, c_flat = array("q"), array("q")
        for text, label in zip(texts, y, strict=True):
            c = class_idx[label]
            pairs, ctxs = self._pairs(text)
            p_flat.extend(p_idx.setdefault(p, len(p_idx)) * k + c for p in pairs)
            c_flat.extend(c_idx.setdefault(x, len(c_idx)) * k + c for x in ctxs)
        trans = np.bincount(np.frombuffer(p_flat, dtype=np.int64), minlength=len(p_idx) * k).reshape(-1, k)
        ctx = np.bincount(np.frombuffer(c_flat, dtype=np.int64), minlength=len(c_idx) * k).reshape(-1, k)
        a, V = self.alpha, self.vocab_size_
        self.log_trans_ = np.vstack([np.log(trans + a), np.full((1, k), math.log(a))])
        self.log_ctx_ = np.vstack([np.log(ctx + a * V), np.full((1, k), math.log(a * V))])
        return self

    def decision_function(self, texts):
        texts = list(texts)
        p_get, c_get = self.pair_index_.get, self.ctx_index_.get
        p_unk, c_unk = len(self.pair_index_), len(self.ctx_index_)
        out = np.empty((len(texts), len(self.classes_)), dtype=np.float64)
        for s in range(0, len(texts), CHUNK):
            block = texts[s : s + CHUNK]
            p_ids, c_ids, lengths = array("q"), array("q"), np.empty(len(block), dtype=np.int64)
            for i, text in enumerate(block):
                pairs, ctxs = self._pairs(text)
                lengths[i] = len(pairs)
                p_ids.extend(p_get(p, p_unk) for p in pairs)
                c_ids.extend(c_get(x, c_unk) for x in ctxs)
            p_ids, c_ids = np.frombuffer(p_ids, dtype=np.int64), np.frombuffer(c_ids, dtype=np.int64)
            ll = _segment_sums(self.log_trans_, p_ids, lengths) - _segment_sums(self.log_ctx_, c_ids, lengths)
            n = np.maximum(1, lengths)[:, None]
            out[s : s + len(block)] = ll / n + self.log_priors_ / n
        return out


class CharMarkov(_SequenceLM):
    START, END, UNK = "\u0002", "\u0003", "\u0004"

    def __init__(self, alpha=0.1, order=2):
        super().__init__(alpha)
        self.order = int(order)

    def _prepare_vocab(self, texts):
        self.alphabet_ = {ch for t in texts for ch in _norm(t)} | {self.START, self.END, self.UNK}
        self.vocab_size_ = len(self.alphabet_)

    def _pairs(self, text):
        alphabet, unk, o = self.alphabet_, self.UNK, self.order
        seq = self.START * o + "".join(ch if ch in alphabet else unk for ch in _norm(text)) + self.END
        rng = range(o, len(seq))
        return [seq[p - o : p + 1] for p in rng], [seq[p - o : p] for p in rng]


class WordMarkov(_SequenceLM):
    START, END, UNK = "<s>", "</s>", "<unk>"

    def __init__(self, alpha=0.1, min_count=3, order=3, tokenizer="regex", entity_min_count=None):
        super().__init__(alpha)
        self.min_count, self.order, self.tokenizer, self.entity_min_count = (
            int(min_count),
            int(order),
            tokenizer,
            entity_min_count,
        )
        self.context_size = self.order - 1

    @staticmethod
    def _is_entity(tok):
        return tok[:1] in ("@", "#") and len(tok) > 1

    def _raw_tokens(self, text):
        toks = list(tweet_tokens(text)) if self.tokenizer == "tweet" else TOKEN_RE.findall(_norm(text))
        if self.entity_min_count is not None:  # se conservan las entidades; se decide en _encode
            return ["URL" if is_url(t) else t for t in toks]
        out = []
        for t in toks:
            if is_url(t):
                out.append("URL")
            elif t.startswith("@") and (self.tokenizer == "regex" or len(t) > 1):
                out.append("@USER")
            elif t.startswith("#") and (self.tokenizer == "regex" or len(t) > 1):
                out.append("#TAG")
            else:
                out.append(t)
        return out

    def _prepare_vocab(self, texts):
        counts = {}
        for t in texts:
            for tok in self._raw_tokens(t):
                counts[tok] = counts.get(tok, 0) + 1
        ent = self.entity_min_count
        self.vocab_ = {
            tok
            for tok, n in counts.items()
            if n >= (ent if ent is not None and self._is_entity(tok) else self.min_count)
        }
        self.vocab_.update(
            (self.START, self.END, self.UNK, "@USER", "#TAG")
            if ent is not None
            else (self.START, self.END, self.UNK)
        )
        self.vocab_size_ = len(self.vocab_)

    def _encode(self, tok):
        if tok in self.vocab_:
            return tok
        if self.entity_min_count is not None and self._is_entity(tok):
            return "@USER" if tok[0] == "@" else "#TAG"
        return self.UNK

    def _pairs(self, text):
        k = self.context_size
        seq = [self.START] * k + [self._encode(t) for t in self._raw_tokens(text)] + [self.END]
        rng = range(k, len(seq))
        return [tuple(seq[p - k : p + 1]) for p in rng], [tuple(seq[p - k : p]) for p in rng]


m = WordMarkov(tokenizer="tweet", entity_min_count=5).fit(X[:2000], y[:2000])
print([m._encode(t) for t in m._raw_tokens("RT @usuario_raro: hola a todos http://t.co/x #algo")])

**c) Kneser-Ney interpolado modificado** (Chen y Goodman, 1999). Es el cambio que más mejoró el modelo final.

El suavizado aditivo le da a todo n-grama no visto la misma probabilidad pequeña. Kneser-Ney, en cambio, descuenta una cantidad fija $D$ a cada n-grama visto y reparte esa masa según un modelo con **un carácter menos de contexto**: si no conozco `xjaj`, miro qué viene después de `jaj`, y si tampoco, después de `aj`. En los órdenes menores no cuenta cuántas veces aparece cada n-grama, sino **en cuántos contextos distintos** aparece (conteos de continuación):

$$P_{KN}(w \mid h) = \frac{\max(c(hw) - D_{c(hw)},\,0)}{c(h)} + \gamma(h)\,P_{KN}(w \mid h'), \qquad \gamma(h) = \frac{D_1 N_1(h\cdot) + D_2 N_2(h\cdot) + D_{3+} N_{3+}(h\cdot)}{c(h)}$$

Los descuentos $D_1, D_2, D_{3+}$ se estiman por orden y por clase a partir de cuántos n-gramas aparecen 1, 2, 3 y 4 veces.

In [ ]:
def _discounts(T):
    # Descuentos (3, k) de Chen y Goodman a partir de los conteos de conteos de cada clase.
    n = [np.maximum((T == i).sum(axis=0), 1).astype(float) for i in (1, 2, 3, 4)]
    Y = n[0] / (n[0] + 2 * n[1])
    D = np.vstack([1 - 2 * Y * n[1] / n[0], 2 - 3 * Y * n[2] / n[1], 3 - 4 * Y * n[3] / n[2]])
    return np.clip(D, 0.05, np.array([[1.0], [2.0], [3.0]]))


class KNCharLM:
    # LM de caracteres con Kneser-Ney interpolado modificado, uno por clase.
    # order = tamaño del n-grama (contexto de order - 1 caracteres).
    # Puntaje = (Σ log P(x_t | contexto) + log P(c)) / n, como los demás LMs secuenciales.
    START, END = "\x02", "\x03"

    def __init__(self, order=6, case_sensitive=True):
        self.order, self.case_sensitive = int(order), bool(case_sensitive)

    def _seq(self, text):
        t = "" if text is None else str(text)
        return self.START * (self.order - 1) + (t if self.case_sensitive else t.casefold()) + self.END

    def _grams(self, seq, n):
        # n-gramas que terminan en cada posición predicha (una por carácter + fin).
        return [seq[p - n + 1 : p + 1] for p in range(self.order - 1, len(seq))]

    @staticmethod
    def _disc(c, D):
        return np.where(c >= 3, D[2], np.where(c == 2, D[1], np.where(c == 1, D[0], 0.0)))

    def fit(self, texts, y):
        texts, y = list(texts), np.asarray(y)
        self.classes_, sizes = np.unique(y, return_counts=True)
        self.log_priors_ = np.log(sizes / sizes.sum())
        cidx = {c: i for i, c in enumerate(self.classes_)}
        k, N = len(self.classes_), self.order
        # conteos crudos del orden mayor
        index, flat = {}, array("q")
        for text, label in zip(texts, y, strict=True):
            c = cidx[label]
            setdef = index.setdefault
            flat.extend(setdef(g, len(index)) * k + c for g in self._grams(self._seq(text), N))
        keys = list(index)
        T = (
            np.bincount(np.frombuffer(flat, dtype=np.int64), minlength=len(keys) * k)
            .reshape(-1, k)
            .astype(np.float32)
        )
        del flat
        self.index_, self.T_, self.hist_, self.D_ = {N: index}, {N: T}, {}, {}
        # órdenes menores: conteos de continuación
        for n in range(N, 1, -1):
            sub_index = {}
            m = np.fromiter(
                (sub_index.setdefault(g[1:], len(sub_index)) for g in keys), dtype=np.int64, count=len(keys)
            )
            pos = (self.T_[n] > 0).astype(np.float32)
            Tn1 = np.stack(
                [np.bincount(m, weights=pos[:, c], minlength=len(sub_index)) for c in range(k)], axis=1
            )
            del m, pos
            self._build_hist(n, keys, sub_index)  # historias g[:-1] en el mismo índice del orden n-1
            if len(sub_index) > len(Tn1):
                Tn1 = np.vstack([Tn1, np.zeros((len(sub_index) - len(Tn1), k))])
            self.index_[n - 1], self.T_[n - 1] = sub_index, Tn1.astype(np.float32)
            keys = list(sub_index)
        # orden 1: interpolado con la uniforme
        T1 = self.T_[1]
        D1 = _discounts(T1)
        self.D_[1] = D1
        tot = T1.sum(axis=0)
        gam = (D1[0] * (T1 == 1).sum(0) + D1[1] * (T1 == 2).sum(0) + D1[2] * (T1 >= 3).sum(0)) / tot
        V = len(self.index_[1]) + 1
        self.P1_ = np.vstack(
            [(np.maximum(T1 - self._disc(T1, D1), 0) / tot) + gam / V, gam / V]
        )  # última fila = no visto
        return self

    def _build_hist(self, n, keys, hidx):
        # Para el orden n: índice de historias, total por historia, numerador de γ y descuentos.
        T = self.T_[n]
        k = T.shape[1]
        D = _discounts(T)
        h = np.fromiter((hidx.setdefault(g[:-1], len(hidx)) for g in keys), dtype=np.int64, count=len(keys))
        H = len(hidx)
        tot = np.stack([np.bincount(h, weights=T[:, c], minlength=H) for c in range(k)], axis=1)
        gnum = np.zeros((H, k))
        for j, mask in enumerate((T == 1, T == 2, T >= 3)):
            gnum += D[j] * np.stack(
                [np.bincount(h, weights=mask[:, c].astype(float), minlength=H) for c in range(k)], 1
            )
        self.hist_[n] = (
            hidx,
            np.vstack([tot, np.zeros((1, k))]).astype(np.float32),
            np.vstack([gnum, np.zeros((1, k))]).astype(np.float32),
        )
        self.D_[n] = D
        self.T_[n] = np.vstack([T, np.zeros((1, k), dtype=np.float32)])  # fila extra = n-grama no visto

    def decision_function(self, texts):
        texts = list(texts)
        N, k = self.order, len(self.classes_)
        out = np.empty((len(texts), k))
        for s in range(0, len(texts), CHUNK):
            seqs = [self._seq(t) for t in texts[s : s + CHUNK]]
            lengths = np.array([len(sq) - (N - 1) for sq in seqs], dtype=np.int64)
            idx1, u1 = self.index_[1], len(self.index_[1])
            ids = np.fromiter((idx1.get(g, u1) for sq in seqs for g in self._grams(sq, 1)), dtype=np.int64)
            p = self.P1_[ids]
            for n in range(2, N + 1):  # se sube de orden: cada nivel interpola con el anterior
                gi, gu = self.index_[n], len(self.index_[n])
                hidx, tot, gnum = self.hist_[n]
                hu = len(hidx)
                grams = [g for sq in seqs for g in self._grams(sq, n)]
                gid = np.fromiter((gi.get(g, gu) for g in grams), dtype=np.int64, count=len(grams))
                hid = np.fromiter((hidx.get(g[:-1], hu) for g in grams), dtype=np.int64, count=len(grams))
                c, t, gn = self.T_[n][gid], tot[hid], gnum[hid]
                seen = t > 0
                p = np.where(
                    seen, (np.maximum(c - self._disc(c, self.D_[n]), 0) + gn * p) / np.where(seen, t, 1.0), p
                )
            ll = _segment_sums(np.log(np.clip(p, 1e-300, None)), np.arange(len(p)), lengths)
            nn = np.maximum(1, lengths)[:, None]
            out[s : s + len(seqs)] = ll / nn + self.log_priors_ / nn
        return out

### 2.3 Canales y caché

Un **canal** es un modelo con su configuración. Su salida son probabilidades por clase:

$$p_k(c \mid x) = \operatorname{softmax}_c\big(\text{puntaje}_k(x, c) / T_k\big)$$

La temperatura $T_k$ ajusta qué tan "seguro" es el canal antes de mezclarlo. Los NB con muchos eventos por tuit dan probabilidades casi 0/1.

**Caché.** Las probabilidades de cada canal se guardan en `cache/` con una huella de la configuración y de los datos. Si cambia cualquiera de las dos, se recalculan. La caché la crea el propio notebook; no hay que subir nada.

In [ ]:
CANALES = {
    # --- mezcla inicial del equipo (0.60021 en Kaggle) ---
    "word1": (NgramNB, dict(unit="word", orders=(1,), alpha=0.01), 1.0),
    "char4": (NgramNB, dict(unit="char", orders=(4,), alpha=0.1), 1.0),
    "char5": (NgramNB, dict(unit="char", orders=(5,), alpha=0.1), 1.0),
    "char6": (NgramNB, dict(unit="char", orders=(6,), alpha=0.03), 1.0),
    "mentions": (IdentityLM, dict(pattern=r"@\w+", alpha=0.1), 1.0),
    "hashtags": (IdentityLM, dict(pattern=r"#\w+", alpha=0.1), 1.0),
    "position": (PositionalWordLM, dict(alpha=0.03), 1.0),
    "mention_pos": (PositionMentionLM, dict(alpha=0.01), 1.0),
    "char_seq2": (CharMarkov, dict(alpha=0.1, order=2), 0.05),
    "word_seq3": (WordMarkov, dict(alpha=0.1, min_count=3, order=3), 0.2),
    # --- greedy_ent: TweetTokenizer, mayúsculas, forma, Markov de caracteres orden 4, LM de entidades ---
    "word1_tt": (NgramNB, dict(unit="word", orders=(1,), alpha=0.01, tokenizer="tweet"), 1.0),
    "position_tt": (PositionalWordLM, dict(alpha=0.03, tokenizer="tweet"), 1.0),
    "mention_pos_tt": (PositionMentionLM, dict(alpha=0.01, tokenizer="tweet"), 1.0),
    "word_seq3_ent5_tt": (
        WordMarkov,
        dict(alpha=0.1, min_count=3, order=3, tokenizer="tweet", entity_min_count=5),
        0.2,
    ),
    "char4_cs": (NgramNB, dict(unit="char", orders=(4,), alpha=0.1, case_sensitive=True), 1.0),
    "char5_cs": (NgramNB, dict(unit="char", orders=(5,), alpha=0.1, case_sensitive=True), 1.0),
    "char6_cs": (NgramNB, dict(unit="char", orders=(6,), alpha=0.03, case_sensitive=True), 1.0),
    "shape7": (ShapeNB, dict(orders=(7,), alpha=0.1), 1.0),
    "char_seq4": (CharMarkov, dict(alpha=0.1, order=4), 1.0),
    # --- greedy_kn: Kneser-Ney de caracteres ---
    "kn_char6_cs": (KNCharLM, dict(order=6, case_sensitive=True), 1.0),
    "kn_char7_cs": (KNCharLM, dict(order=7, case_sensitive=True), 1.0),
    "kn_char6": (KNCharLM, dict(order=6, case_sensitive=False), 1.0),
}


def softmax(scores):
    scores = scores - scores.max(axis=1, keepdims=True)
    e = np.exp(scores)
    return e / e.sum(axis=1, keepdims=True)


def entrenar_canal(nombre, X_tr, y_tr, X_ev):
    cls, kwargs, T = CANALES[nombre]
    modelo = cls(**kwargs).fit(X_tr, y_tr)
    scores = modelo.decision_function(X_ev)
    orden = [int(np.flatnonzero(modelo.classes_ == c)[0]) for c in CLASES]
    del modelo
    gc.collect()
    return nombre, softmax(scores[:, orden] / T)


def _huella(*partes):
    h = hashlib.sha1()
    for parte in partes:
        for item in parte if isinstance(parte, (list, tuple, np.ndarray)) else [parte]:
            h.update(str(item).encode("utf-8"))
            h.update(b"\x1f")
        h.update(b"\x1e")
    return h.hexdigest()[:16]


def probs_canales(nombres, X_tr, y_tr, X_ev, etiqueta, n_jobs=N_JOBS):
    # Carga de la caché lo que ya existe y entrena el resto (en paralelo con n_jobs > 1).
    datos = _huella(X_tr, y_tr, X_ev)
    rutas = {
        n: CACHE
        / f"{etiqueta}__{n}__{_huella(CANALES[n][0].__name__, sorted(CANALES[n][1].items()), CANALES[n][2], datos)}.npy"
        for n in nombres
    }
    out = {n: np.load(r) for n, r in rutas.items() if r.exists()}
    faltan = [n for n in nombres if n not in out]
    if faltan:
        t0 = time.perf_counter()
        print(f"[{etiqueta}] entrenando {len(faltan)} canal(es): {', '.join(faltan)}")
        # Primero los KN (los más lentos), para repartir mejor el trabajo entre procesos.
        faltan.sort(key=lambda n: CANALES[n][0] is not KNCharLM)
        if n_jobs == 1:
            nuevos = dict(entrenar_canal(n, X_tr, y_tr, X_ev) for n in faltan)
        else:
            import multiprocessing as mp
            from concurrent.futures import ProcessPoolExecutor

            with ProcessPoolExecutor(
                max_workers=min(n_jobs, len(faltan)), mp_context=mp.get_context("fork")
            ) as ex:
                nuevos = dict(
                    ex.map(
                        entrenar_canal,
                        faltan,
                        [X_tr] * len(faltan),
                        [y_tr] * len(faltan),
                        [X_ev] * len(faltan),
                    )
                )
        for n, p in nuevos.items():
            np.save(rutas[n], p)
        out.update(nuevos)
        print(f"[{etiqueta}] listo en {time.perf_counter() - t0:.0f} s")
    return {n: out[n] for n in nombres}

### 2.4 Las tres mezclas que se comparan

Los canales se combinan con una **mezcla lineal de probabilidades** construida paso a paso. Cada canal nuevo entra con un peso $w$ sobre la mezcla anterior: $(1-w)\,p_{\text{prev}} + w\,p_k$. La temperatura y el peso de cada paso se eligieron en una partición de **ajuste** (semilla 20261108). Se aceptaban solo si también mejoraban en una partición de **confirmación** (semilla 20261109) con un intervalo bootstrap al 95 % por encima de 0.

1. **Mezcla inicial** (Kaggle 0.60021). NB de palabras y de caracteres 4/5/6, más hashtags, menciones (T = 0.35, peso 0.30), posición (0.10), menciones con posición (0.30, solo en tuits con @), Markov de caracteres orden 2 (0.15) y Markov de palabras orden 3 (0.20).
2. **`greedy_ent`** (Kaggle 0.60796). La misma estructura con cuatro reemplazos:
   - TweetTokenizer en los canales de palabras;
   - caracteres 4/5/6 **con mayúsculas**;
   - LM de palabras **con entidades** (menciones y hashtags frecuentes).

   Además agrega forma (T = 1, peso 0.05) y Markov de caracteres orden 4 (T = 0.1, peso 0.15).
3. **`greedy_kn`** (Kaggle **0.61369**). `greedy_ent` con tres movimientos Kneser-Ney, aceptados por la búsqueda greedy:
   - `char6_cs` → `kn_char6_cs` (T = 0.2, en la misma posición);
   - `char_seq4` → `kn_char7_cs` (T = 0.2, peso 0.30);
   - se agrega `kn_char6` sin mayúsculas (T = 1, peso 0.40).

Así quedan los pesos efectivos en `greedy_kn`: mezcla base ≈ 40 %, `kn_char6` 40 %, `kn_char7_cs` 18 % y forma 2 %.

In [ ]:
def con_temperatura(p, T):
    return softmax(np.log(np.clip(p, 1e-300, 1.0)) / T)


def agregar(base, q, T, w):
    return (1 - w) * base + w * con_temperatura(q, T)


def mezcla_base(p, menciones):
    # Mezcla inicial. p: canal lógico -> probabilidades; menciones: máscara de tuits con @.
    out = 0.216 * p["word1"] + 0.144 * p["char4"] + 0.240 * p["char5"] + 0.400 * p["char6"]
    out = (0.36 * out + 0.40 * p["hashtags"]) / 0.76
    out = 0.70 * out + 0.30 * softmax(np.log(np.clip(p["mentions"], 1e-12, 1.0)) / 0.35)
    out = 0.90 * out + 0.10 * p["position"]
    out = out.copy()
    out[menciones] = 0.70 * out[menciones] + 0.30 * p["mention_pos"][menciones]
    out = 0.85 * out + 0.15 * p["char_seq2"]
    return 0.80 * out + 0.20 * p["word_seq3"]


# Cada modelo: canal lógico -> (canal real, temperatura extra) y canales agregados (nombre, T, peso) en orden.
_BASE = {
    n: (n, None)
    for n in [
        "word1",
        "char4",
        "char5",
        "char6",
        "mentions",
        "hashtags",
        "position",
        "mention_pos",
        "char_seq2",
        "word_seq3",
    ]
}
_ENT = {
    **_BASE,
    "word1": ("word1_tt", None),
    "position": ("position_tt", None),
    "mention_pos": ("mention_pos_tt", None),
    "word_seq3": ("word_seq3_ent5_tt", None),
    "char4": ("char4_cs", None),
    "char5": ("char5_cs", None),
    "char6": ("char6_cs", None),
}
MODELOS = {
    "mezcla inicial": (_BASE, []),
    "greedy_ent": (_ENT, [("shape7", 1.0, 0.05), ("char_seq4", 0.1, 0.15)]),
    "greedy_kn": (
        {**_ENT, "char6": ("kn_char6_cs", 0.2)},
        [("shape7", 1.0, 0.05), ("kn_char7_cs", 0.2, 0.30), ("kn_char6", 1.0, 0.40)],
    ),
}


def canales_de(modelo):
    mapa, adds = MODELOS[modelo]
    return list(dict.fromkeys([real for real, _ in mapa.values()] + [a[0] for a in adds]))


def predecir(modelo, probs, menciones):
    mapa, adds = MODELOS[modelo]
    p = {
        log: (probs[real] if T is None else con_temperatura(probs[real], T))
        for log, (real, T) in mapa.items()
    }
    out = mezcla_base(p, menciones)
    for nombre, T, w in adds:
        out = agregar(out, probs[nombre], T, w)
    return out


for nombre in MODELOS:
    print(f"{nombre:<15} {len(canales_de(nombre))} canales: {', '.join(canales_de(nombre))}")

## 3. Validación

**Partición sin fuga.** `StratifiedGroupKFold` con 5 folds: un fold (≈ 20 %) es validación y el resto es train. El **grupo** es el texto normalizado (sin espacios en los extremos y en minúsculas), así que todas las copias de un mismo tuit quedan del mismo lado. La celda verifica que ningún texto aparezca en ambos lados.

Se usa la semilla de **confirmación** (20261109), la misma de las cifras del informe. Los parámetros de las mezclas se eligieron en la otra semilla (20261108).

In [ ]:
if VALIDAR:
    grupos = np.array([t.strip().casefold() for t in X])
    folds = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEMILLA_VALIDACION)
    idx_tr, idx_va = next(folds.split(X, y, groups=grupos))
    assert not set(grupos[idx_tr]) & set(grupos[idx_va]), "hay textos en train y validación"
    X_tr, y_tr = [X[i] for i in idx_tr], y[idx_tr]
    X_va, y_va = [X[i] for i in idx_va], y[idx_va]
    print(f"train {len(X_tr):,} | validación {len(X_va):,} | textos compartidos: 0")

In [ ]:
if VALIDAR:
    todos = list(dict.fromkeys(c for m in MODELOS for c in canales_de(m)))
    probs_va = probs_canales(todos, X_tr, y_tr, X_va, etiqueta=f"val{SEMILLA_VALIDACION}")
    menciones_va = mention_mask(X_va)

### 3.1 Cada canal por sí solo

F1 macro de cada canal usado solo (gana la clase más probable). Se ve por qué el modelo final se apoya en los LMs de caracteres. Kneser-Ney (`kn_*`) es el mejor canal individual; conservar mayúsculas (`_cs`) mejora a su versión en minúsculas. Los canales sociales (menciones, hashtags) son débiles solos porque solo informan en los tuits que los tienen, pero aportan mucho en la mezcla.

In [ ]:
K = len(CLASES)
IDX = {c: i for i, c in enumerate(CLASES)}


def a_idx(etiquetas):
    return np.fromiter((IDX[c] for c in etiquetas), dtype=np.int64, count=len(etiquetas))


def _f1_cm(cm):
    tp = np.diag(cm).astype(float)
    prec = np.divide(tp, cm.sum(0), out=np.zeros(K), where=cm.sum(0) > 0)
    rec = np.divide(tp, cm.sum(1), out=np.zeros(K), where=cm.sum(1) > 0)
    return np.divide(2 * prec * rec, prec + rec, out=np.zeros(K), where=(prec + rec) > 0)


def matriz(y_idx, pred):
    return np.bincount(y_idx * K + pred, minlength=K * K).reshape(K, K)


def f1_macro(y_idx, pred):
    return float(_f1_cm(matriz(y_idx, pred)).mean())


def bootstrap_delta(y_idx, pred_a, pred_b, n_boot=N_BOOT, seed=0):
    # Intervalo al 95 % de F1(b) - F1(a), remuestreando las mismas filas para ambos modelos.
    rng = np.random.default_rng(seed)
    ca, cb, n = y_idx * K + pred_a, y_idx * K + pred_b, len(y_idx)
    d = np.empty(n_boot)
    for i in range(n_boot):
        r = rng.integers(0, n, n)
        d[i] = (
            _f1_cm(np.bincount(cb[r], minlength=K * K).reshape(K, K)).mean()
            - _f1_cm(np.bincount(ca[r], minlength=K * K).reshape(K, K)).mean()
        )
    return np.percentile(d, [2.5, 97.5])


if VALIDAR:
    yv = a_idx(y_va)
    solo = pd.Series(
        {n: f1_macro(yv, p.argmax(1)) for n, p in probs_va.items()}, name="F1 macro (canal solo)"
    )
    display(solo.sort_values(ascending=False).round(4).to_frame())

### 3.2 Comparación de los tres modelos

F1 macro en validación y en el leaderboard público. Con todos los datos, la columna `= informe` debe dar `True`: confirma que el notebook reproduce los modelos enviados.

In [ ]:
F1_INFORME = {"mezcla inicial": 0.5869, "greedy_ent": 0.5937, "greedy_kn": 0.5992}
KAGGLE = {"mezcla inicial": 0.60021, "greedy_ent": 0.60796, "greedy_kn": 0.61369}

if VALIDAR:
    pv = {m: predecir(m, probs_va, menciones_va) for m in MODELOS}
    pred_va = {m: p.argmax(1) for m, p in pv.items()}
    f1_va = {m: f1_macro(yv, p) for m, p in pred_va.items()}
    display(
        pd.DataFrame(
            {
                "F1 validación": f1_va,
                "Kaggle público": KAGGLE,
                "= informe": {m: abs(f1_va[m] - F1_INFORME[m]) < 5e-4 for m in MODELOS},
            }
        ).round({"F1 validación": 4})
    )

### 3.3 `greedy_kn` frente a `greedy_ent`

Las dos submissions finales, comparadas cara a cara en los mismos tuits de validación:

- la tabla da el F1 de cada clase y la diferencia;
- debajo, el intervalo bootstrap del Δ y cuántos tuits cambian de predicción entre los dos modelos.

In [ ]:
if VALIDAR:
    pe, pk_ = pred_va["greedy_ent"], pred_va["greedy_kn"]
    cara = pd.DataFrame(
        {"greedy_ent": _f1_cm(matriz(yv, pe)), "greedy_kn": _f1_cm(matriz(yv, pk_))}, index=CLASES
    )
    cara.loc["F1 macro"] = cara.mean()
    cara["Δ (kn − ent)"] = cara["greedy_kn"] - cara["greedy_ent"]
    display(cara.round(4))

    lo, hi = bootstrap_delta(yv, pe, pk_)
    corrige, dana = ((pe != yv) & (pk_ == yv)).sum(), ((pe == yv) & (pk_ != yv)).sum()
    print(f"Δ F1 macro {f1_va['greedy_kn'] - f1_va['greedy_ent']:+.4f}  IC95 [{lo:+.4f}, {hi:+.4f}]")
    print(
        f"Cambia la predicción en {(pe != pk_).mean():.1%} de los tuits: greedy_kn corrige {corrige:,} y daña {dana:,} (neto {corrige - dana:+,})"
    )

**Qué se ve.** Kneser-Ney mejora **todas las clases**, no solo una. El intervalo del Δ queda por encima de 0 y la ganancia se confirmó en Kaggle (+0.0057). Los dos modelos cambian la predicción en ≈ 9 % de los tuits y `greedy_kn` gana más de los que pierde. Las clases extremas (13-17, 50-64, 65-xx) son las más fáciles. Las centrales (18-24 y 25-34) siguen siendo las débiles: 25-34 no tiene vocabulario propio, es una clase de transición.

In [ ]:
if VALIDAR:
    cm = matriz(yv, pred_va["greedy_kn"])
    cmn = cm / cm.sum(1, keepdims=True)
    fig, ax = plt.subplots(figsize=(5.2, 4.6))
    ax.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
    for i in range(K):
        for j in range(K):
            ax.text(
                j,
                i,
                f"{cmn[i, j]:.2f}",
                ha="center",
                va="center",
                color="white" if cmn[i, j] > 0.5 else "black",
                fontsize=9,
            )
    ax.set_xticks(range(K), CLASES, rotation=45)
    ax.set_yticks(range(K), CLASES)
    ax.set_xlabel("predicha")
    ax.set_ylabel("real")
    ax.set_title("greedy_kn: matriz de confusión (cada fila suma 1)")
    plt.tight_layout()
    plt.show()

### 3.4 Análisis de errores

**Errores ordinales.** Las clases tienen orden, así que un error a la clase vecina es menos grave que confundir 13-17 con 65-xx.

In [ ]:
if VALIDAR:
    pk = pv["greedy_kn"]
    pred = pred_va["greedy_kn"]
    err = pred != yv
    dist = np.abs(pred - yv)
    print(f"Tuits mal clasificados: {err.mean():.1%}")
    print(
        f"Errores en la clase vecina: {(dist[err] == 1).mean():.1%} | a distancia ≤ 2: {(dist[err] <= 2).mean():.1%}"
    )
    print(f"Tuits con error de 3 o más rangos: {(dist >= 3).mean():.1%}")

    # Accuracy por tipo de tuit (un tuit puede estar en varios grupos)
    sv = pd.Series(X_va)
    propio = (
        sv.str.replace(r"(?i)^rt\b|https?://\S+|www\.\S+|@\w+|#\w+|[^\w]", " ", regex=True)
        .str.strip()
        .str.len()
        == 0
    )
    tipos = {
        "todos": np.ones(len(sv), bool),
        "0-30 caracteres": (sv.str.len() <= 30).to_numpy(),
        "sin texto propio": propio.to_numpy(),
        "RT": sv.str.match(r"(?i)^rt @").to_numpy(),
        "respuesta (empieza con @)": sv.str.match(r"^@\w+").to_numpy(),
        "con hashtag": sv.str.contains(r"#\w+").to_numpy(),
        "con URL": sv.str.contains(r"https?://|www\.").to_numpy(),
    }
    display(
        pd.DataFrame(
            {
                t: {
                    "% de los tuits": 100 * m.mean(),
                    "% de los errores": 100 * (err & m).sum() / err.sum(),
                    "accuracy": (~err[m]).mean(),
                }
                for t, m in tipos.items()
            }
        ).T.round(2)
    )

    # Confianza: margen entre la 1.ª y la 2.ª clase, por quintiles
    top2 = np.sort(pk, axis=1)[:, -2:]
    margen = top2[:, 1] - top2[:, 0]
    q = pd.qcut(margen, 5, labels=["Q1 (más dudoso)", "Q2", "Q3", "Q4", "Q5 (más seguro)"])
    display(
        pd.Series(~err)
        .groupby(q, observed=True)
        .mean()
        .rename("accuracy por quintil de margen")
        .round(3)
        .to_frame()
    )

**Qué se ve.** La mayoría de los errores caen en la clase vecina, que es la frontera natural de una variable continua cortada en rangos. Los tuits cortos, los que no tienen texto propio (solo URL, RT o menciones) y las plantillas concentran los errores graves. El margen entre las dos clases más probables predice bien el acierto: en el quintil más seguro acierta más del 95 %.

**Ejemplos reales mal clasificados** (sorteados con semilla fija): tres errores con alta confianza y tres entre clases vecinas centrales.

In [ ]:
if VALIDAR:
    rng = np.random.default_rng(SEMILLA_EJEMPLOS)
    conf = pk.max(1)
    cats = {
        "alta confianza (≥ 0.6)": np.flatnonzero(err & (conf >= 0.6)),
        "18-24 ↔ 25-34": np.flatnonzero(err & np.isin(yv, [1, 2]) & np.isin(pred, [1, 2])),
    }
    filas = []
    for cat, idx in cats.items():
        for i in rng.choice(idx, size=min(3, len(idx)), replace=False):
            top = np.argsort(-pk[i])[:3]
            filas.append(
                {
                    "tipo": cat,
                    "tuit": X_va[i][:110],
                    "real": CLASES[yv[i]],
                    "predicha": CLASES[pred[i]],
                    "top-3": ", ".join(f"{CLASES[j]} {pk[i, j]:.2f}" for j in top),
                }
            )
    display(pd.DataFrame(filas))

**Lectura de los ejemplos.**

- **Empates entre vecinas.** Muchos errores tienen la 1.ª y la 2.ª clase a pocos puntos: el tuit no tiene pistas que separen 18-24 de 25-34.
- **Retuits.** En un RT, la cuenta mencionada es el autor original, no a quien se le habla. La señal social apunta entonces a la edad de quienes suelen retuitear esa cuenta, no a la del autor.
- **Estilo engañoso.** Fandom, emoticonos o inglés empujan hacia clases jóvenes aunque el autor sea mayor.

### 3.5 Alternativas que se probaron y se descartaron

Los experimentos completos están en el informe. Se descartaron porque no mejoraron en las dos particiones:

| Alternativa | Δ F1 en confirmación |
|---|---|
| Filtrar de train las plantillas ruidosas | −0.0038 |
| Sesgos por clase en la decisión | +0.0044 en ajuste, −0.0002 en confirmación (sobreajuste) |
| Mezclas conjuntas optimizadas por log-loss | −0.0091 a −0.0024 |
| Mezcla log-lineal con pesos optimizados para F1 | 0.5977 < 0.5992 |
| Interpolación Jelinek-Mercer en el LM de palabras | −0.0022 |
| Markov en reversa, canal de longitud, especialista 18-24 vs 25-34 | ≈ 0 |
| NB binarizado y Kneser-Ney de palabras | mejoran solos, pero no al entrar en la mezcla |

## 4. Modelo final: entrenamiento con todo el train y submission

Se entrenan con **todo** `age_train.csv` los canales de las dos submissions finales y se predice `age_test.csv`. `greedy_ent` comparte 11 canales con `greedy_kn` y solo agrega `char6_cs` y `char_seq4`. Cada modelo escribe un CSV con las columnas `id` y `age_range`:

| Archivo | Modelo | Kaggle público |
|---|---|---|
| `submission_greedy_kn.csv` | `greedy_kn` (principal) | 0.61369 |
| `submission_greedy_ent.csv` | `greedy_ent` (respaldo) | 0.60796 |

In [ ]:
t0 = time.perf_counter()
FINALES = ["greedy_kn", "greedy_ent"]
probs_test = probs_canales(
    list(dict.fromkeys(c for m in FINALES for c in canales_de(m))), X, y, X_test, etiqueta="final"
)
menciones_test = mention_mask(X_test)
print(f"Canales listos en {time.perf_counter() - t0:.0f} s")

subs = {}
for m in FINALES:
    pred_test = np.asarray(CLASES)[predecir(m, probs_test, menciones_test).argmax(1)]
    sub = pd.DataFrame({"id": test["id"], "age_range": pred_test})
    assert len(sub) == len(test) and sub["id"].is_unique and sub["age_range"].isin(CLASES).all()
    sub.to_csv(WORK / f"submission_{m}.csv", index=False, encoding="utf-8")
    subs[m] = sub["age_range"]
    print("Guardado:", WORK / f"submission_{m}.csv")

print(
    f"Los dos modelos coinciden en el {(subs['greedy_kn'] == subs['greedy_ent']).mean():.1%} de los tuits de test."
)
f1_local = f1_va if VALIDAR else F1_INFORME  # sin validación, se muestra el F1 reportado en el informe
resumen = pd.DataFrame({m: s.value_counts().reindex(CLASES) for m, s in subs.items()}).T
resumen["F1 local"] = [round(f1_local[m], 4) for m in FINALES]
resumen["Kaggle público"] = [KAGGLE[m] for m in FINALES]
resumen

## 5. Resultados y conclusiones

| Modelo | F1 validación (confirmación) | Kaggle público |
|---|---|---|
| Mezcla inicial | 0.5869 | 0.60021 |
| + LM de entidades | 0.5890 | 0.60270 |
| `greedy_conservador_ent` | 0.5929 | 0.60532 |
| `greedy_ent` | 0.5937 | 0.60796 |
| **`greedy_kn`** | **0.5992** | **0.61369** |

La relación entre las mejoras locales y las de Kaggle fue cercana a 1:1 en todos los envíos (razón 0.86–1.16).

**Conclusiones.**

1. Solo con LMs de n-gramas propios se llegó a 0.614 en el leaderboard público, +0.013 sobre la mezcla inicial, con mejora en las seis clases.
2. Las mejoras que funcionaron salieron del análisis de datos:
   - conservar las mayúsculas;
   - modelar como entidades las menciones y hashtags frecuentes;
   - usar un mejor suavizado para los n-gramas largos (Kneser-Ney), que fue el cambio individual más rentable.
3. Las técnicas con más grados de libertad (sesgos por clase, mezclas conjuntas, mezcla log-lineal) sobreajustaron: ganaban en ajuste y perdían en confirmación.
4. El error que queda es mayoritariamente entre clases vecinas y en tuits con poco texto propio, un límite de clasificar con un solo tuit.

**Limitación de la validación.** La partición de confirmación también se consultó al aceptar cada paso greedy, así que su F1 es algo optimista. La concordancia con Kaggle sugiere que el sesgo es pequeño.

## Referencias y licencias

- Chen, S. F. y Goodman, J. (1999). An empirical study of smoothing techniques for language modeling. *Computer Speech & Language*, 13(4).
- Kneser, R. y Ney, H. (1995). Improved backing-off for M-gram language modeling. *ICASSP*.
- Peng, F., Schuurmans, D. y Wang, S. (2003). Language and task independent text categorization with simple language models. *HLT-NAACL*.
- Monroe, B. L., Colaresi, M. P. y Quinn, K. M. (2008). Fightin' Words. *Political Analysis*, 16(4).
- Moreno-Sandoval, L. G. et al. (2018). Age classification from Spanish tweets. *ICEIS*.
- Bibliotecas:
  - NLTK, solo `TweetTokenizer` (Apache 2.0);
  - NumPy, pandas, SciPy y scikit-learn (BSD-3); de scikit-learn solo se usa `StratifiedGroupKFold`;
  - Matplotlib (licencia Matplotlib, basada en PSF).